# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 3: Evaluation, Baselines, Traditional ML

Today we'll write some simple models to predict the price of a product

We'll use an approach to evaluate the performance of the model

And we'll test some Baseline Models using Traditional machine learning

匯入本日所需的所有套件：數值運算（`numpy`）、資料處理（`pandas`）、機器學習模型與評估工具（`sklearn`），以及課程自定的 `evaluate`（評估函式）和 `Item`（商品資料結構）。

In [1]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

In [2]:
LITE_MODE = False

`LITE_MODE = False` 選擇使用完整資料集（約 80 萬筆），設為 `True` 可切換至輕量版（約 2 萬筆）快速測試。

接著從 Hugging Face Hub 載入由 Day 1/2 整理完成、已含 `summary` 欄位的資料集，切分為訓練集、驗證集與測試集三份。後續所有模型都在**同一份測試集**上評估，確保比較公平。

In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

README.md:   0%|          | 0.00/744 [00:00<?, ?B/s]

c:\Users\XBOT\Documents\projects\llm_engineering_unofficial\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\XBOT\.cache\huggingface\hub\datasets--ed-donner--items_full. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. 

train-00000-of-00001.parquet:   0%|          | 0.00/243M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


validation-00000-of-00001.parquet:   0%|          | 0.00/3.04M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/3.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


定義第一個「愚蠢基線」：對每件商品隨機猜一個 $1–$999 的價格，建立「任何有意義的模型都應該比它好」的最低下限。用 `random.seed(42)` 固定亂數種子，確保結果可重現。

In [4]:
def random_pricer(item):
    return random.randrange(1,1000)

`evaluate` 函式對測試集（200 筆）逐一預測，以顏色標示準確度（綠色＝接近、紅色＝差距大），並計算整體指標：**Error**（平均絕對誤差）、**MSE**（均方誤差）、**R²**（解釋變異量）。這三個指標貫穿本日所有模型的比較。

In [17]:
random.seed(42)
evaluate(random_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$436 $1 $29 $690 $252 $21 $85 $72 $719 $225 $20 $380 $894 $505 $11 $572 $354 $17 $179 $23 $90 $115 $433 $442 $304 $122 $291 $714 $567 $639 $539 $370 $66 $380 $489 $534 $769 $835 $207 $740 $626 $84 $680 $178 $129 $260 $142 $189 $836 $580 $310 $25 $380 $270 $47 $234 $861 $313 $417 $259 $591 $33 $657 $361 $79 $38 $757 $500 $263 $5 $534 $284 $570 $625 $584 $871 $759 $361 $575 $178 $602 $60 $17 $579 $207 $732 $115 $224 $756 $193 $866 $9 $370 $250 $456 $423 $821 $217 $103 $195 $264 $98 $650 $135 $470 $842 $675 $264 $43 $325 $591 $138 $516 $619 $56 $2 $449 $369 $221 $845 $640 $616 $501 $59 $502 $273 $844 $688 $616 $81 $164 $705 $52 $795 $259 $396 $70 $2 $89 $798 $902 $331 $818 $716 $129 $186 $627 $2 $141 $873 $918 $553 $423 $7 $253 $136 $204 $707 $255 $502 $19 $739 $557 $426 $80 $575 $39 $321 $185 $132 $497 $484 $326 $751 $53 $733 $85 $64 $549 $71 $158 $672 $133 $362 $16 $373 $258 $544 $420 $515 $223 $944 $532 $743 $866 $68 $527 $459 $67 $673 

> **結果：Error $382.08 | MSE 219,084 | R² -896.9%**  
> 完全隨機猜測，誤差極大。作為「最差基準」，後續任何有學習能力的模型都應遠優於此。

定義第二個基線：計算訓練集所有商品的**平均價格**（約 $140.57），不論傳入什麼商品都一律回傳此固定值。這是「完全不看商品資訊」時能做到的最佳預測。

In [ ]:
# That was fun!
# We can do better - here's another rather trivial model

training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

In [ ]:
evaluate(constant_pricer, test)

> **結果：Error $106.18 | MSE 22,011 | R² -0.2%**  
> 比隨機猜好很多，但 R² ≈ 0 代表幾乎沒有解釋能力。這是後續所有模型必須超越的「最低門檻」。

### 傳統機器學習：結構化特徵線性迴歸

從商品資料萃取三個**結構化特徵**：
- `weight`：商品重量（磅）
- `weight_unknown`：重量是否缺失（缺失時為 1，區分「重量為零」與「資料未填」）
- `text_length`：摘要文字長度（字元數）

`weight_unknown` 是常見的**二元旗標（Binary Flag）**技巧，避免模型將缺失值誤判為有意義的零。

In [ ]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [ ]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

將訓練集與測試集的特徵連同目標欄位（`price`）整合成 pandas DataFrame，再用三個結構化特徵訓練**線性迴歸**，並印出各特徵係數，衡量每個特徵對預測的貢獻大小。

In [ ]:
# Traditional Linear Regression!

np.random.seed(42)

# Separate features and target
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Train a Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# Predict the test set and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

**係數解讀**：`weight +0.449`（越重越貴）、`weight_unknown -6.63`（重量缺失略降價）、`text_length +0.247`（描述越長略貴）；截距 $51.1。

R² = **-5.9%**，比常數定價器更差！說明這三個特徵的線性組合解釋能力非常有限，需要引入更豐富的特徵——商品的**文字內容**本身才是關鍵。

In [ ]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

將訓練好的線性迴歸包成統一介面的定價函式，方便直接傳入 `evaluate`。

In [ ]:
evaluate(linear_regression_pricer, test)

> **結果：Error $101.56 | MSE 20,832 | R² 5.2%**  
> 比常數定價器略好，但進步有限。說明只憑重量和文字長度遠遠不夠，需要讓模型直接讀懂文字描述。

### 詞袋模型（Bag of Words）：讓機器讀懂文字

將訓練集的摘要文字（`documents`）與對應價格（`prices`）分別抽出，準備進行詞袋向量化。`CountVectorizer` 從所有摘要中選出 **2000 個最常見詞**（排除 "the"、"a" 等停用詞），將每段摘要轉成長度為 2000 的**稀疏詞頻向量**。

此向量矩陣 `X` 將被後續所有文字模型（線性迴歸、隨機森林、XGBoost）共用，不需重複計算。

In [ ]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)


In [ ]:
# Here are the 1,000 most common words that it picked, not including "stop words":

selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

列出被選入的 2000 個詞，直觀確認向量化結果合理（排除無意義停用詞）。

以詞袋向量 `X` 為特徵、商品價格 `prices` 為目標，訓練**文字版線性迴歸**。模型學習「哪些詞出現 → 價格高低」的線性關係，取代原本人工設計的三個結構化特徵。

In [ ]:
regressor = LinearRegression()
regressor.fit(X, prices)

包成定價函式，用 `max(..., 0)` 確保輸出**不為負數**（線性迴歸在極端詞組合下可能預測出負價格）。

In [ ]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [ ]:
evaluate(natural_language_linear_regression_pricer, test)

> **結果：Error $76.81 | MSE 12,786 | R² 41.8%**  
> 比結構化特徵線性迴歸大幅改善，R² 躍升至 41.8%，驗證了商品文字描述含有豐富的價格資訊。

### 隨機森林（Random Forest）訓練

取前 **15,000 筆**子集訓練（完整資料需約 15 小時），使用 100 棵決策樹取平均預測。每棵樹以不同的資料子集與特徵子集訓練，避免單棵樹的過擬合，並捕捉詞袋特徵中的**非線性關係**（線性迴歸做不到這點）。

In [ ]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

## Random Forest model

The Random Forest is a type of "**ensemble**" algorithm, meaning that it combines many smaller algorithms to make better predictions.

It uses a very simple kind of machine learning algorithm called a **decision tree**. A decision tree makes predictions by examining the values of features in the input. Like a flow chart with IF statements. Decision trees are very quick and simple, but they tend to overfit.

In our case, the "features" are the elements of the Vector - in other words, it's the number of times that a particular word appears in the product description.

So you can think of it something like this:

**Decision Tree**  
\- IF the word "TV" appears more than 3 times THEN  
-- IF the word "LED" appears more than 2 times THEN  
--- IF the word "HD" appears at least once THEN  
---- Price = $500


With Random Forest, multiple decision trees are created. Each one is trained with a different random subset of the data, and a different random subset of the features. You can see above that we specify 100 trees, which is the default.

Then the Random Forest model simply takes the average of all its trees to product the final result.

In [ ]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [ ]:
evaluate(random_forest, test)

> **結果：Error $73.04 | MSE 12,747 | R² 42.0%**  
> 比文字線性迴歸略好，集成決策樹能捕捉部分非線性關係。若使用完整 80 萬筆資料，誤差可進一步降至約 **$56.40**。

In [ ]:
# This is how to save the model if you want to, particularly if you run this on a larger dataset

# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

## Introducing XGBoost

Like Random Forest, XGBoost is also an ensemble model that combines multiple decision trees.

But unlike Random Forest, XGBoost builds one tree after another, with each next tree correcting for errors in the prior trees, using 'gradient descent'.

It's much faster than Random Forest, so we can run it for the full dataset, and it's typically better at generalizing.

**If this import doesn't work, please skip this! It's not required. On a Mac, you might need to do `brew install libomp` in the terminal.**



In [ ]:
import xgboost as xgb

In [ ]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

`n_estimators=1000` 棵樹，`learning_rate=0.1` 控制每棵新樹的修正幅度（越小越保守但越精確）。因為 XGBoost 速度遠快於隨機森林，此處可使用**完整訓練集**，而非子集。

In [ ]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

包成定價函式，同樣用 `max(0, ...)` 確保預測值不為負數。

In [ ]:
evaluate(xg_boost, test)

> **結果：Error $68.23 | MSE 9,582 | R² 56.4%**  
> **本日最佳！** 梯度提升的序列修正機制，加上使用完整資料集，使 XGBoost 明顯優於隨機森林。R² 達 56.4%，模型能解釋超過一半的價格變異。
>
> 各模型誤差比較：隨機 $382 → 常數 $106 → 結構化線性迴歸 $102 → 文字線性迴歸 $77 → 隨機森林 $73 → XGBoost **$68**
>
> 下一步（Day 4）：用神經網路與 Frontier LLM 繼續挑戰更低的誤差。

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">Traditional ML isn't just useful for learning the history; it's still heavily used in industry today, particularly for tasks where there are clearly identifiable features. It's worth spending time exploring the algorithms and experimenting here. See if you can beat my numbers with Traditional ML! I ran the Random Forest for the entire 800,000 training dataset. It took about 15 hours to run, and it ended up getting a low error of $56.40. Traditional ML can do well - try it for yourself.</span>
        </td>
    </tr>
</table>